# 06 — Dual Encoder Training (Task 48a) — Final Version

**Architecture:** `VulBERTaFusionModel` from `05d_dual_encoder_model.ipynb`
- GraphCodeBERT + LoRA (trainable)
- VulBERTa `claudios/VulBERTa-MLP-D2A` loaded via RobertaModel (frozen)
- FusionProjectionLayer: concat(1536) → Linear → LayerNorm → GELU → (768)
- CWE head: Linear(768→384) → GELU → Dropout → Linear(384→8)

**Caches:**
- `tokens_maxlen512.pt`  — GraphCodeBERT tokens, 14,522 chunks
- `tokens_vulberta_maxlen512.pt` — VulBERTa tokens, 14,522 chunks

**Starting weights:** `vulberta_fusion_init.pt` from `05d`

**Colab protection:** saves to Drive every 100 steps. On reconnect: re-run all cells → auto-resumes.

**Deliverable for Sohaila (task 49a):**
- `checkpoints/dual_encoder_best.pt`
- `logs/dual_encoder_metrics.csv`

## 0 — GPU Check

**Run this first. Do NOT proceed on CPU.**

In [48]:
import torch, sys

print('=' * 55)
print('GPU CHECK')
print('=' * 55)

if not torch.cuda.is_available():
    print('NO GPU -- Runtime > Disconnect and delete runtime > Reconnect')
    raise SystemExit('Stopping -- no GPU')

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'  GPU  : {gpu_name}')
print(f'  VRAM : {vram_gb:.1f} GB')

if vram_gb > 30:
    BATCH_SIZE = 16
    print(f'  A100/V100 -- batch_size={BATCH_SIZE}  ~20 min/epoch')
elif vram_gb > 12:
    BATCH_SIZE = 8
    print(f'  T4 -- batch_size={BATCH_SIZE}  ~40-50 min/epoch')
else:
    BATCH_SIZE = 4
    print(f'  Small GPU -- batch_size={BATCH_SIZE}  ~80 min/epoch')

print('Good to go.')
print('=' * 55)


GPU CHECK
  GPU  : Tesla T4
  VRAM : 15.6 GB
  T4 -- batch_size=8  ~40-50 min/epoch
Good to go.


## 1 — Install

In [49]:
import subprocess, sys
# Remove torchao -- causes PEFT import error
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'],
               check=False, capture_output=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'peft>=0.10', 'torch',
    'pyyaml', 'scikit-learn', 'tqdm',
], check=False)
print('Done.')


Done.


## 2 — Paths, Config & Device

In [50]:
import os, sys, json, random, platform, csv, time, subprocess, shutil
from pathlib import Path
import yaml
import torch
import numpy as np

if 'BATCH_SIZE' not in dir():
    BATCH_SIZE = 8

IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# ── Auto-find BASE_DIR by locating config.yaml ────────────────────────────
print('Searching for project files on Drive...')
r = subprocess.run(['find', '/content/drive', '-name', 'config.yaml',
                    '-not', '-path', '*/.*'],
                   capture_output=True, text=True, timeout=60)
candidates = [p.strip() for p in r.stdout.strip().split('\n') if p.strip()]
print(f'  config.yaml found at: {candidates}')

BASE_DIR = None
for c in candidates:
    base = Path(c).parent
    if (base / 'datasets').exists() or (base / 'checkpoints').exists():
        BASE_DIR = base
        break
if BASE_DIR is None and candidates:
    BASE_DIR = Path(candidates[0]).parent

assert BASE_DIR is not None, 'Cannot find project on Drive. Make sure files are shared.'
print(f'  BASE_DIR: {BASE_DIR}')

cfg_path = BASE_DIR / 'config.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

TOKEN_CACHE_DIR = BASE_DIR / cfg['token_cache_dir']
CHECKPOINT_DIR  = BASE_DIR / cfg['checkpoint_dir']
LOG_DIR         = BASE_DIR / cfg['log_dir']
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

MAX_LEN        = 512
GCB_CACHE      = TOKEN_CACHE_DIR / 'tokens_maxlen512.pt'
VULBERTA_CACHE = TOKEN_CACHE_DIR / 'tokens_vulberta_maxlen512.pt'
INIT_CKPT      = CHECKPOINT_DIR  / 'vulberta_fusion_init.pt'
BEST_CKPT      = CHECKPOINT_DIR  / 'dual_encoder_best.pt'
METRICS_CSV    = LOG_DIR          / 'dual_encoder_metrics.csv'

# ── Auto-find dual_encoder_last.pt (handles shortcuts + shared drives) ────
r2 = subprocess.run(['find', '/content/drive', '-name', 'dual_encoder_last.pt'],
                    capture_output=True, text=True, timeout=60)
last_found = [p.strip() for p in r2.stdout.strip().split('\n') if p.strip()]

if last_found:
    LAST_CKPT = Path(last_found[0])
    print(f'  dual_encoder_last.pt: {LAST_CKPT}')
else:
    LAST_CKPT = CHECKPOINT_DIR / 'dual_encoder_last.pt'
    print(f'  dual_encoder_last.pt: not found yet')

# ── If last.pt is corrupted (epoch=0), copy best.pt over it ──────────────
if LAST_CKPT.exists():
    try:
        tmp = torch.load(LAST_CKPT, map_location='cpu', weights_only=False)
        saved_epoch = tmp.get('epoch', 0)
        saved_f1    = tmp.get('best_val_f1', 0.0)
        del tmp
        if saved_epoch == 0 or saved_f1 == 0.0:
            print(f'  last.pt is corrupted (epoch={saved_epoch}, f1={saved_f1}) -- replacing with best.pt')
            r3 = subprocess.run(['find', '/content/drive', '-name', 'dual_encoder_best.pt'],
                                capture_output=True, text=True, timeout=60)
            best_found = [p.strip() for p in r3.stdout.strip().split('\n') if p.strip()]
            if best_found:
                best_src = Path(best_found[0])
                best_tmp = torch.load(best_src, map_location='cpu', weights_only=False)
                best_epoch = best_tmp.get('epoch', 0)
                best_f1    = best_tmp.get('best_val_f1', 0.0)
                del best_tmp
                if best_epoch > 0:
                    shutil.copy(str(best_src), str(LAST_CKPT))
                    print(f'  Copied best.pt (epoch={best_epoch}, f1={best_f1:.4f}) → last.pt')
                else:
                    print(f'  best.pt also corrupted -- will start fresh')
                    LAST_CKPT = Path('/nonexistent')  # force fresh start
            else:
                print('  best.pt not found -- will start fresh')
                LAST_CKPT = Path('/nonexistent')
        else:
            print(f'  last.pt OK: epoch={saved_epoch}, f1={saved_f1:.4f}')
    except Exception as e:
        print(f'  last.pt unreadable: {e} -- will try best.pt')
        LAST_CKPT = Path('/nonexistent')

# ── Pre-flight checks ─────────────────────────────────────────────────────
print('\nPre-flight checks:')
all_ok = True
for label, path in [
    ('GCB cache (512)',       GCB_CACHE),
    ('VulBERTa cache (512)',  VULBERTA_CACHE),
    ('config.yaml',          cfg_path),
]:
    exists = path.exists()
    size   = f'{path.stat().st_size/1e6:.0f} MB' if exists else 'MISSING'
    print(f'  [{"OK" if exists else "MISSING":7}] {label:30s} {size}')
    if not exists: all_ok = False

# checkpoint is optional -- just warn
ckpt_exists = LAST_CKPT.exists()
print(f'  [{"OK" if ckpt_exists else "WARN   "}] last checkpoint              {"will resume" if ckpt_exists else "fresh start"}')

assert all_ok, 'Fix missing cache files first.'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    torch.backends.cudnn.benchmark = True

SEED = cfg.get('seed', 42)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if DEVICE == 'cuda': torch.cuda.manual_seed_all(SEED)
print(f'\nDevice     : {DEVICE}')
print(f'Batch size : {BATCH_SIZE}')
print(f'MAX_LEN    : {MAX_LEN}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab -- BASE_DIR: /content/drive/MyDrive/Colab Notebooks/CSI_Project
Pre-flight checks:
  [OK     ] GCB cache (Anas 512)           120 MB
  [OK     ] VulBERTa cache 512             120 MB
  [OK     ] vulberta_fusion_init.pt        1008 MB
  [OK     ] config.yaml                    0 MB
Device     : cuda
Batch size : 8
MAX_LEN    : 512


## 3 — Hyperparameters

In [51]:
# Must match 05d_dual_encoder_model.ipynb exactly
GCB_MODEL_NAME  = 'microsoft/graphcodebert-base'
VULB_MODEL_NAME = 'claudios/VulBERTa-MLP-D2A'
HIDDEN_DIM      = 768
NUM_CWE_CLASSES = 8    # 7 CWE types + unknown (index 7)
MAX_LEN         = 512
DROPOUT_RATE    = 0.1
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05
LORA_TARGETS = ['query', 'key', 'value']  # match latest.pt

NUM_EPOCHS       = 30
LR_LORA          = 2e-5
LR_NEW           = 1e-4
WEIGHT_DECAY     = 0.01
WARMUP_RATIO     = 0.10
MAX_GRAD_NORM    = 1.0
SAVE_EVERY_STEPS = 100
EARLY_STOP_PAT   = 3

# 8 classes
CWE_LABEL_MAP = {
    'CWE-077': 0, 'CWE-601': 1, 'CWE-022': 2, 'CWE-094': 3,
    'CWE-089': 4, 'CWE-352': 5, 'CWE-079': 6, 'unknown': 7,
}
CWE_NAMES = [
    'CWE-077', 'CWE-601', 'CWE-022', 'CWE-094',
    'CWE-089', 'CWE-352', 'CWE-079', 'unknown'
]

print('Hyperparameters:')
print(f'  VULB_MODEL_NAME  : {VULB_MODEL_NAME}')
print(f'  NUM_CWE_CLASSES  : {NUM_CWE_CLASSES}')
print(f'  MAX_LEN          : {MAX_LEN}')
print(f'  NUM_EPOCHS       : {NUM_EPOCHS}')
print(f'  BATCH_SIZE       : {BATCH_SIZE}')
print(f'  LR_LORA / LR_NEW : {LR_LORA} / {LR_NEW}')
print(f'  SAVE_EVERY_STEPS : {SAVE_EVERY_STEPS}')


Hyperparameters:
  VULB_MODEL_NAME  : claudios/VulBERTa-MLP-D2A
  NUM_CWE_CLASSES  : 8
  MAX_LEN          : 512
  NUM_EPOCHS       : 15
  BATCH_SIZE       : 8
  LR_LORA / LR_NEW : 2e-05 / 0.0001
  SAVE_EVERY_STEPS : 100


## 4 — Model Definition (exact copy from `05d_dual_encoder_model.ipynb`)

In [52]:
import torch.nn as nn
from transformers import AutoModel, RobertaModel
from peft import LoraConfig, get_peft_model, TaskType
from typing import Optional, Dict


class FusionProjectionLayer(nn.Module):
    def __init__(self, hidden_dim=HIDDEN_DIM, proj_dim=HIDDEN_DIM, dropout=DROPOUT_RATE):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(hidden_dim * 2, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        nn.init.xavier_uniform_(self.projection[0].weight)
        nn.init.zeros_(self.projection[0].bias)

    def forward(self, gcb_cls, vulb_cls):
        return self.projection(torch.cat([gcb_cls, vulb_cls], dim=-1))


class VulBERTaFusionModel(nn.Module):
    def __init__(self, gcb_model_name=GCB_MODEL_NAME,
                 vulb_model_name=VULB_MODEL_NAME,
                 num_labels=NUM_CWE_CLASSES, hidden_dim=HIDDEN_DIM,
                 dropout=DROPOUT_RATE, freeze_vulberta=True):
        super().__init__()

        # Encoder 1: GraphCodeBERT + LoRA
        _gcb  = AutoModel.from_pretrained(gcb_model_name)
        _lora = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGETS, bias='none',
        )
        self.graphcodebert = get_peft_model(_gcb, _lora)

        # Encoder 2: VulBERTa (frozen) -- use RobertaModel to strip classifier head
        self.vulberta = RobertaModel.from_pretrained(
            vulb_model_name,
            ignore_mismatched_sizes=True,
        )
        if freeze_vulberta:
            for p in self.vulberta.parameters():
                p.requires_grad = False

        self.fusion = FusionProjectionLayer(hidden_dim, hidden_dim, dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_labels),
        )
        # No ignore_index -- all labels 0-7 are valid
        self.loss_fn = nn.CrossEntropyLoss()

    @staticmethod
    def _get_cls(encoder, input_ids, attention_mask):
        return encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        ).last_hidden_state[:, 0, :]

    def forward(self, gcb_input_ids, gcb_attention_mask,
                vulb_input_ids, vulb_attention_mask, labels=None):
        gcb_cls  = self._get_cls(self.graphcodebert, gcb_input_ids,  gcb_attention_mask)
        vulb_cls = self._get_cls(self.vulberta,      vulb_input_ids, vulb_attention_mask)
        fused    = self.fusion(gcb_cls, vulb_cls)
        logits   = self.classifier(fused)
        out = {'logits': logits, 'fused_repr': fused}
        if labels is not None:
            out['loss'] = self.loss_fn(logits, labels)
        return out


print('VulBERTaFusionModel defined')
print(f'  GCB    : {GCB_MODEL_NAME} + LoRA (trainable)')
print(f'  VulB   : {VULB_MODEL_NAME} via RobertaModel (frozen)')
print(f'  Classes: {NUM_CWE_CLASSES}')


VulBERTaFusionModel defined
  GCB    : microsoft/graphcodebert-base + LoRA (trainable)
  VulB   : claudios/VulBERTa-MLP-D2A via RobertaModel (frozen)
  Classes: 8


## 5 — Cache-based Dataset & DataLoaders

In [53]:
from torch.utils.data import Dataset, DataLoader


class DualCacheDataset(Dataset):
    """
    Loads from Anas's GCB cache + VulBERTa cache.
    Both caches: 14,522 records (chunks of 4,085 functions), max_len=512.
    """
    def __init__(self, gcb_cache, vulb_cache, split):
        assert gcb_cache['num_records'] == vulb_cache['num_records'], \
            'Cache size mismatch -- rebuild with 05c'
        assert gcb_cache['split_origins'] == vulb_cache['split_origins'], \
            'Cache alignment mismatch -- rebuild with 05c'

        indices = (
            list(range(len(gcb_cache['split_origins'])))
            if split == 'all' else
            [i for i, s in enumerate(gcb_cache['split_origins']) if s == split]
        )
        self.gcb_ids   = gcb_cache['input_ids'][indices]
        self.gcb_mask  = gcb_cache['attention_mask'][indices]
        self.vulb_ids  = vulb_cache['input_ids'][indices]
        self.vulb_mask = vulb_cache['attention_mask'][indices]
        self.labels    = gcb_cache['cwe_labels'][indices]

    def __len__(self): return len(self.gcb_ids)

    def __getitem__(self, idx):
        return {
            'gcb_input_ids':       self.gcb_ids[idx],
            'gcb_attention_mask':  self.gcb_mask[idx],
            'vulb_input_ids':      self.vulb_ids[idx],
            'vulb_attention_mask': self.vulb_mask[idx],
            'label':               self.labels[idx],
        }


print('Loading caches into RAM...')
gcb_cache  = torch.load(GCB_CACHE,      weights_only=True)
vulb_cache = torch.load(VULBERTA_CACHE, weights_only=True)
print(f'  GCB cache    : {tuple(gcb_cache["input_ids"].shape)}')
print(f'  VulBERTa     : {tuple(vulb_cache["input_ids"].shape)}')

train_ds = DualCacheDataset(gcb_cache, vulb_cache, split='train')
val_ds   = DualCacheDataset(gcb_cache, vulb_cache, split='val')
print(f'  Train samples: {len(train_ds):,}')
print(f'  Val samples  : {len(val_ds):,}')

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, persistent_workers=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True, persistent_workers=True
)

steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * NUM_EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)
print(f'  Steps/epoch  : {steps_per_epoch}')
print(f'  Total steps  : {total_steps}')
print(f'  Warmup steps : {warmup_steps}')


Loading caches into RAM...
  GCB cache    : (14522, 512)
  VulBERTa     : (14522, 512)
  Train samples: 12,994
  Val samples  : 1,528
  Steps/epoch  : 1625
  Total steps  : 24375
  Warmup steps : 2437


## 6 — Load Model + Optimizer + Scheduler

Auto-resumes from `dual_encoder_last.pt` if it exists.
Otherwise loads `vulberta_fusion_init.pt` from `05d` and starts fresh.

In [54]:
from transformers import get_linear_schedule_with_warmup

print('Building VulBERTaFusionModel...')
model = VulBERTaFusionModel(freeze_vulberta=True).to(DEVICE)

total_p     = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Total     : {total_p:,}')
print(f'  Trainable : {trainable_p:,} ({100*trainable_p/total_p:.2f}%)')

lora_params = [p for n, p in model.named_parameters()
               if p.requires_grad and 'graphcodebert' in n]
new_params  = [p for n, p in model.named_parameters()
               if p.requires_grad and ('fusion' in n or 'classifier' in n)]

optimizer = torch.optim.AdamW([
    {'params': lora_params, 'lr': LR_LORA},
    {'params': new_params,  'lr': LR_NEW},
], weight_decay=WEIGHT_DECAY)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps,
)

start_epoch = 0
global_step = 0
best_val_f1 = 0.0
no_improve  = 0

if LAST_CKPT.exists():
    print(f'\nResuming from: {LAST_CKPT.name}')
    ckpt = torch.load(LAST_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch = ckpt['epoch']
    global_step = ckpt['global_step']
    best_val_f1 = ckpt.get('best_val_f1', 0.0)
    no_improve  = ckpt.get('no_improve',  0)
    print(f'  Resumed from epoch : {start_epoch}')
    print(f'  Global step        : {global_step}')
    print(f'  Best val F1        : {best_val_f1:.4f}')
    print(f'  No improve counter : {no_improve}/{EARLY_STOP_PAT}')
else:
    print('\nNo checkpoint -- loading fresh weights')
    init_state = torch.load(INIT_CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(
        init_state if not isinstance(init_state, dict) or 'model_state' not in init_state
        else init_state['model_state']
    )
    SINGLE_CKPT  = CHECKPOINT_DIR / 'latest.pt'
    single       = torch.load(SINGLE_CKPT, map_location=DEVICE, weights_only=False)
    single_state = single.get('model_state', single.get('model_state_dict', single))
    remapped = {}
    for k, v in single_state.items():
        if k.startswith('encoder.') and 'lora' in k:
            remapped[k.replace('encoder.', 'graphcodebert.', 1)] = v
        elif k.startswith('graphcodebert.') and 'lora' in k:
            remapped[k] = v
    model.load_state_dict(remapped, strict=False)
    print(f'  LoRA transferred from latest.pt')

print(f'\nStarting from epoch {start_epoch + 1} / {NUM_EPOCHS}')

Building VulBERTaFusionModel...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: claudios/VulBERTa-MLP-D2A
Key                        | Status     | 
---------------------------+------------+-
classifier.out_proj.weight | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
classifier.dense.weight    | UNEXPECTED | 
pooler.dense.bias          | MISSING    | 
pooler.dense.weight        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  Total     : 251,846,024
  Trainable : 2,365,064 (0.94%)

Resuming from: dual_encoder_last.pt
  Resumed from epoch : 9
  Global step        : 14625
  Best val F1        : 0.6630
  No improve counter : 3/3

Starting from epoch 10 / 15


## 7 — Helper Functions

In [55]:
from sklearn.metrics import f1_score, precision_score, recall_score


def save_checkpoint(path, epoch, global_step, best_val_f1, no_improve, is_best=False):
    ckpt = {
        'epoch':           epoch,
        'global_step':     global_step,
        'best_val_f1':     best_val_f1,
        'no_improve':      no_improve,
        'model_state':     model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
    }
    torch.save(ckpt, path)
    if is_best:
        torch.save(ckpt, BEST_CKPT)


def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            gcb_ids   = batch['gcb_input_ids'].to(DEVICE)
            gcb_mask  = batch['gcb_attention_mask'].to(DEVICE)
            vulb_ids  = batch['vulb_input_ids'].to(DEVICE)
            vulb_mask = batch['vulb_attention_mask'].to(DEVICE)
            labels    = batch['label']
            out       = model(gcb_ids, gcb_mask, vulb_ids, vulb_mask)
            preds     = out['logits'].argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.numpy().tolist())
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    p  = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    r  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    return f1, p, r, np.array(all_preds), np.array(all_labels)


def log_metrics(epoch, train_loss, val_f1, val_p, val_r):
    exists = METRICS_CSV.exists()
    with open(METRICS_CSV, 'a', newline='') as f:
        w = csv.writer(f)
        if not exists:
            w.writerow(['epoch','train_loss','val_macro_f1',
                        'val_precision','val_recall'])
        w.writerow([epoch, f'{train_loss:.4f}', f'{val_f1:.4f}',
                    f'{val_p:.4f}', f'{val_r:.4f}'])


print('Helper functions defined: save_checkpoint, evaluate, log_metrics')


Helper functions defined: save_checkpoint, evaluate, log_metrics


## 8 — Training Loop

On Colab disconnect: re-run ALL cells from top → auto-resumes from last checkpoint.

In [ ]:
from tqdm import tqdm
import math

# Safety check -- if resume failed in cell 14, catch it here
if start_epoch == 0 and LAST_CKPT.exists():
    print('Safety reload: start_epoch=0 but checkpoint exists -- reloading...')
    ckpt = torch.load(LAST_CKPT, map_location=DEVICE, weights_only=False)
    saved_epoch = ckpt.get('epoch', 0)
    if saved_epoch > 0:
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        scheduler.load_state_dict(ckpt['scheduler_state'])
        start_epoch = saved_epoch
        global_step = ckpt['global_step']
        best_val_f1 = ckpt.get('best_val_f1', 0.0)
        no_improve  = ckpt.get('no_improve', 0)
        print(f'  Reloaded: epoch={start_epoch}, f1={best_val_f1:.4f}')
    else:
        print('  Checkpoint epoch=0 -- starting fresh')

print('=' * 60)
print('DUAL ENCODER TRAINING  --  Task 48a')
print('=' * 60)
print(f'Epochs        : {start_epoch + 1} to {NUM_EPOCHS}')
print(f'Steps/epoch   : {steps_per_epoch}')
print(f'Batch size    : {BATCH_SIZE}')
print(f'LR LoRA / New : {LR_LORA} / {LR_NEW}')
print(f'Save every    : {SAVE_EVERY_STEPS} steps')
print(f'Early stop    : patience={EARLY_STOP_PAT}  current={no_improve}')
print(f'Device        : {DEVICE}')
print('=' * 60)

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_num   = epoch + 1
    model.train()
    epoch_start = time.time()
    epoch_loss  = 0.0
    valid_steps = 0
    nan_steps   = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch_num}/{NUM_EPOCHS}', leave=True)

    for batch in pbar:
        gcb_ids   = batch['gcb_input_ids'].to(DEVICE)
        gcb_mask  = batch['gcb_attention_mask'].to(DEVICE)
        vulb_ids  = batch['vulb_input_ids'].to(DEVICE)
        vulb_mask = batch['vulb_attention_mask'].to(DEVICE)
        labels    = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        out  = model(gcb_ids, gcb_mask, vulb_ids, vulb_mask, labels=labels)
        loss = out['loss']

        if torch.isnan(loss) or torch.isinf(loss):
            nan_steps   += 1
            global_step += 1
            tqdm.write(f'  [Step {global_step}] WARNING: NaN/Inf skipped')
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            MAX_GRAD_NORM
        )
        optimizer.step()
        scheduler.step()

        epoch_loss  += loss.item()
        valid_steps += 1
        global_step += 1

        pbar.set_postfix({
            'loss'   : f'{loss.item():.4f}',
            'step'   : global_step,
            'lr'     : f'{scheduler.get_last_lr()[0]:.2e}',
            'nan_sk' : nan_steps,
        })

        if global_step % SAVE_EVERY_STEPS == 0:
            save_checkpoint(LAST_CKPT, epoch, global_step, best_val_f1, no_improve)
            tqdm.write(f'  [Step {global_step}] checkpoint saved to Drive')

    epoch_time = time.time() - epoch_start
    avg_loss   = epoch_loss / valid_steps if valid_steps > 0 else float('nan')
    nan_pct    = nan_steps / steps_per_epoch * 100

    print(f'\nEpoch {epoch_num} complete ({epoch_time/60:.1f} min)')
    print(f'  Train loss   : {avg_loss:.4f}  '
          f'(valid={valid_steps}, skipped={nan_steps} [{nan_pct:.1f}%])')

    print('  Running validation...')
    val_f1, val_p, val_r, _, _ = evaluate(model, val_loader)
    print(f'  Val macro F1 : {val_f1:.4f}')
    print(f'  Val precision: {val_p:.4f}')
    print(f'  Val recall   : {val_r:.4f}')

    log_metrics(epoch_num, avg_loss, val_f1, val_p, val_r)

    is_best = val_f1 > best_val_f1
    if is_best:
        best_val_f1 = val_f1
        no_improve  = 0
        print(f'  [NEW BEST] val F1 = {best_val_f1:.4f} -- saved to {BEST_CKPT.name}')
    else:
        no_improve += 1
        print(f'  No improvement ({no_improve}/{EARLY_STOP_PAT})')

    save_checkpoint(LAST_CKPT, epoch_num, global_step,
                    best_val_f1, no_improve, is_best=is_best)

    if no_improve >= EARLY_STOP_PAT:
        print(f'\nEarly stopping after {no_improve} epochs without improvement.')
        print(f'Best val macro F1: {best_val_f1:.4f}')
        break

    model.train()

print('\n' + '=' * 60)
print('TRAINING COMPLETE')
print(f'Best val macro F1 : {best_val_f1:.4f}')
print(f'Best checkpoint   : {BEST_CKPT}')
print(f'Metrics CSV       : {METRICS_CSV}')
print('=' * 60)


DUAL ENCODER TRAINING  --  Task 48a
Building VulBERTaFusionModel...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: claudios/VulBERTa-MLP-D2A
Key                        | Status     | 
---------------------------+------------+-
classifier.out_proj.weight | UNEXPECTED | 
classifier.out_proj.bias   | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
classifier.dense.weight    | UNEXPECTED | 
pooler.dense.bias          | MISSING    | 
pooler.dense.weight        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Epoch 1/15:   6%|▌         | 100/1625 [01:36<1:55:08,  4.53s/it, loss=1.8801, step=100, lr=8.21e-07, nan_sk=0]

  [Step 100] checkpoint saved to Drive


Epoch 1/15:  12%|█▏        | 200/1625 [03:46<5:40:58, 14.36s/it, loss=2.1097, step=200, lr=1.64e-06, nan_sk=0]

  [Step 200] checkpoint saved to Drive


Epoch 1/15:  15%|█▍        | 240/1625 [04:20<20:08,  1.15it/s, loss=1.8453, step=240, lr=1.97e-06, nan_sk=0]

## 9 — Final Evaluation

Load best checkpoint and produce final numbers for Sohaila (task 49a).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

assert BEST_CKPT.exists(), f'No best checkpoint -- run training first'
best = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(best['model_state'])
print(f'Loaded best checkpoint  epoch={best["epoch"]}  val_f1={best["best_val_f1"]:.4f}')

val_f1, val_p, val_r, preds, labels = evaluate(model, val_loader)

print('\n' + '=' * 60)
print('DUAL ENCODER FINAL RESULTS')
print('=' * 60)
print(f'  Macro F1        : {val_f1:.4f}')
print(f'  Macro Precision : {val_p:.4f}')
print(f'  Macro Recall    : {val_r:.4f}')
print()
print(classification_report(labels, preds,
      target_names=CWE_NAMES, zero_division=0, digits=3))

cm = confusion_matrix(labels, preds)
print('Confusion matrix (rows=true, cols=predicted):')
header = ''.join(f'{c:>10}' for c in CWE_NAMES)
print(f'{"":>12}{header}')
for i, row in enumerate(cm):
    print(f'{CWE_NAMES[i]:>12}' + ''.join(f'{v:>10}' for v in row))

# Training curves
import csv as csv_module
epochs_l, losses_l, f1s_l = [], [], []
if METRICS_CSV.exists():
    with open(METRICS_CSV) as f:
        for row in csv_module.DictReader(f):
            epochs_l.append(int(row['epoch']))
            losses_l.append(float(row['train_loss']))
            f1s_l.append(float(row['val_macro_f1']))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs_l, losses_l, 'b-o'); ax1.set_title('Train Loss'); ax1.grid(True)
    ax2.plot(epochs_l, f1s_l, 'g-o')
    ax2.axhline(y=0.6304, color='r', linestyle='--', label='Single encoder (0.6304)')
    ax2.set_title('Val Macro F1'); ax2.legend(); ax2.grid(True)
    plt.tight_layout()
    plot_path = LOG_DIR / 'training_curves.png'
    plt.savefig(str(plot_path), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved training curves: {plot_path}')

# GO/NO-GO
SINGLE_ENCODER_F1 = 0.6304
delta = val_f1 - SINGLE_ENCODER_F1
print()
print('=' * 60)
print('GO / NO-GO for Sohaila (task 49a)')
print('=' * 60)
print(f'  Single encoder : {SINGLE_ENCODER_F1:.4f}')
print(f'  Dual encoder   : {val_f1:.4f}')
print(f'  Delta          : {delta:+.4f}')
if delta > 0.02:
    print('  VERDICT: GO -- dual encoder improves by >2%')
elif delta > 0:
    print('  VERDICT: MARGINAL -- small improvement, team decides')
else:
    print('  VERDICT: NO-GO -- did not improve over single encoder')
print()
print(f'Files for Sohaila:')
print(f'  {BEST_CKPT}')
print(f'  {METRICS_CSV}')
